# Load data
Trong phần này, thứ ta cần làm chính là chuyển 2 file data raw thành 1 df thống nhất

In [74]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/raw")


PROJECT_ROOT = find_project_root()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [78]:
df_shoppee = pd.read_json(
    DATA_RAW_DIR / "shopee_reviews_dataset.jsonl",
    lines=True
)
df_shoppee.head(), df_shoppee.shape

(            id                                             review  rating  \
 0  74263765409  Hương vị:thom  Chắc do bên giao hàng bị vỡ mấy...       3   
 1  11104151002  Hương thơm:nhẹ nhàng Lợi ích:phục hồi cấp ẩm M...       5   
 2  15888299382  Chất lượng sản phẩm:ok Đúng với mô tả:đúng  Lầ...       5   
 3  81030214453  Độ tuổi sử dụng:em bes Chất lượng sản phẩm:tot...       5   
 4  88484377297  Hương vị:Mix vị, Tím  Mình mua 64 gói (32 gói ...       1   
 
       label  
 0  negative  
 1  positive  
 2  positive  
 3  positive  
 4  negative  ,
 (9599, 4))

In [80]:
df_aug = pd.read_json(
    DATA_RAW_DIR / "aug_unaccented_reviews.jsonl",
    lines=True
)
df_aug.head(), df_aug.shape

(            id                                             review  rating  \
 0  59241444271  Mui huong:Ngot nhe, thomm Phu hop voi loai da:...       4   
 1  75292434640  Toi rat hai long voi dich vu cua nguoi ban va ...       5   
 2  13364343551  Review nhe mot so dong sua con uong tang can t...       5   
 3  84816190204  Hieu qua:Sach Thiet ke:Dep  Thiet ke nho gon, ...       5   
 4  30102564756  Giao hang nhanh, dong goi dep an toan Mui Pink...       4   
 
       label  
 0  positive  
 1  positive  
 2  positive  
 3  positive  
 4  positive  ,
 (1348, 4))

In [82]:
df_all = pd.concat([df_aug, df_shoppee], ignore_index=True)
df_all.shape

(10947, 4)

# Clean and Normalize text
Ta cần nhớ yêu cầu của 2 mô hình sẽ khác nhau:
- Đối với mô hình LSTM thì yêu cầu đơn giản
- Đối với mô hình baseline cần xử lí kỹ và phức tạp hớn

### Clean
Trong phần clean này, những điều cần làm chung cho cả hai mô hình:
- Bỏ url, html, email
- Chuyển emoji thành text
- Chuẩn hóa khoảng trắng

Riêng đối mới mô hình cho baseline cần xử lí một số thứ đặt biệt hơn
- Lower
- Bỏ dấu câu



In [83]:
import re

EMOJI_SENTIMENT_MAP = {
    # Tích cực
    "😍": "positive_emoji",
    "🥰": "positive_emoji",
    "😘": "positive_emoji",
    "❤️": "positive_emoji",
    "💖": "positive_emoji",
    "💕": "positive_emoji",
    "💝": "positive_emoji",
    "😊": "positive_emoji",
    "😁": "positive_emoji",
    "😄": "positive_emoji",
    "😃": "positive_emoji",
    "😀": "positive_emoji",
    "🙂": "positive_emoji",
    "👍": "positive_emoji",
    "👏": "positive_emoji",
    "✨": "positive_emoji",
    "🌟": "positive_emoji",
    "🔥": "positive_emoji",
    "🎉": "positive_emoji",

    # Tiêu cực
    "😡": "negative_emoji",
    "😠": "negative_emoji",
    "😤": "negative_emoji",
    "😞": "negative_emoji",
    "😔": "negative_emoji",
    "😢": "negative_emoji",
    "😭": "negative_emoji",
    "😥": "negative_emoji",
    "☹️": "negative_emoji",
    "🙁": "negative_emoji",
    "👎": "negative_emoji",
    "💔": "negative_emoji",
    "🤬": "negative_emoji",

    # Hài hước
    "😂": "laugh_emoji",
    "🤣": "laugh_emoji",
    "😆": "laugh_emoji",

    # Bất ngờ
    "😮": "surprise_emoji",
    "😲": "surprise_emoji",
    "😱": "surprise_emoji",

    # Trung tính
    "😐": "neutral_emoji",
    "😑": "neutral_emoji",
    "🤔": "neutral_emoji",

    # Chê bai / mỉa mai
    "🙄": "sarcasm_emoji",
    "😒": "sarcasm_emoji",

    # Tức giận mạnh
    "🤯": "angry_emoji",
    "💢": "angry_emoji",
}

EMOJI_PATTERN = re.compile(
    "|".join(re.escape(emoji) for emoji in EMOJI_SENTIMENT_MAP.keys())
)

def replace_emoji_sentiment(text: str) -> str:
    if not isinstance(text, str):
        return ""

    return EMOJI_PATTERN.sub(
        lambda m: f" {EMOJI_SENTIMENT_MAP[m.group(0)]} ",
        text
    )

def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

def remove_html(text):
    return re.sub(r"<.*?>", "", text)

def remove_email(text: str) -> str:
    if not isinstance(text, str):
        return ""
    email_pattern = r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b'
    return re.sub(email_pattern, '', text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()
    # turn consecutive whitespaces into one, remove begin/end whitespace

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = remove_url(text)
    text = remove_html(text)
    text = normalize_whitespace(text)
    text = replace_emoji_sentiment(text)
    return text

In [86]:
#apply cho cả 2 dataset
df_baseline = df_all.copy()
df_ltsm = df_all.copy()

df_baseline["review"] = df_baseline["review"].apply(clean_text)
df_ltsm["review"] = df_ltsm["review"].apply(clean_text)

df_ltsm.head(), df_baseline.head()

(            id                                             review  rating  \
 0  59241444271  mui huong:ngot nhe, thomm phu hop voi loai da:...       4   
 1  75292434640  toi rat hai long voi dich vu cua nguoi ban va ...       5   
 2  13364343551  review nhe mot so dong sua con uong tang can t...       5   
 3  84816190204  hieu qua:sach thiet ke:dep thiet ke nho gon, t...       5   
 4  30102564756  giao hang nhanh, dong goi dep an toan mui pink...       4   
 
       label  
 0  positive  
 1  positive  
 2  positive  
 3  positive  
 4  positive  ,
             id                                             review  rating  \
 0  59241444271  mui huong:ngot nhe, thomm phu hop voi loai da:...       4   
 1  75292434640  toi rat hai long voi dich vu cua nguoi ban va ...       5   
 2  13364343551  review nhe mot so dong sua con uong tang can t...       5   
 3  84816190204  hieu qua:sach thiet ke:dep thiet ke nho gon, t...       5   
 4  30102564756  giao hang nhanh, dong goi dep an

In [89]:
# Xử lí thêm cho baseline
def remove_punctuation(text):
    return re.sub(r"[^\w\s]", " ", text)

df_baseline["review"] = df_baseline["review"].apply(remove_punctuation)
df_baseline["review"] = df_baseline["review"].str.lower()

df_baseline.head()


,id,review,rating,label
0,59241444271,mui huong ngot nhe thomm phu hop voi loai da ...,4,positive
1,75292434640,toi rat hai long voi dich vu cua nguoi ban va ...,5,positive
2,13364343551,review nhe mot so dong sua con uong tang can t...,5,positive
3,84816190204,hieu qua sach thiet ke dep thiet ke nho gon t...,5,positive
4,30102564756,giao hang nhanh dong goi dep an toan mui pink...,4,positive


# Normalize_text
Trong phần này ta cần:
- Chuyển tiếng Việt không dấu -> có dấu
- Đổi lại các từ viết tắt
- Chuẩn hóa lại các từ bị kéo dài

Riêng với xử lí cho baseline: cần thêm bỏ stopword

In [95]:
import re
# Loại bỏ từ bị kéo dài (lặp lại từ 3 chữ cái trở lên)
def normalize_elongated_words(text: str) -> str:
    if not isinstance(text, str):
        return ""

    return re.sub(r'(.)\1{2,}', r'\1\1', text)



In [96]:
import unicodedata
from collections import Counter, defaultdict

SLANG_DICT = {
    "ko": "không",
    "khum": "không",
    "hok": "không",
    "k": "không",
    "kh": "không",
    "dc": "được",
    "đc": "được",
    "mn": "mọi người",
    "ntn": "như thế nào",
    "sp": "sản phẩm",
    "shop": "shop",
    "ship": "ship",
    "shipper": "shipper",
}

DIACRITIC_OVERRIDES = {
    "a": "à",
    "ai": "ai",
    "anh": "ảnh",
    "ban": "bạn",
    "bao": "bao",
    "bi": "bị",
    "biet": "biết",
    "binh": "bình",
    "bo": "bỏ",
    "cam": "cảm",
    "can": "cần",
    "cao": "cao",
    "chat": "chất",
    "chi": "chỉ",
    "cho": "cho",
    "chong": "chống",
    "chua": "chưa",
    "co": "có",
    "con": "còn",
    "cua": "của",
    "cung": "cũng",
    "da": "da",
    "danh": "đánh",
    "de": "dễ",
    "den": "đến",
    "dep": "đẹp",
    "di": "đi",
    "dich": "dịch",
    "dieu": "điều",
    "dong": "đóng",
    "duoc": "được",
    "gia": "giá",
    "giao": "giao",
    "giup": "giúp",
    "hang": "hàng",
    "hay": "hay",
    "hieu": "hiệu",
    "hoi": "hơi",
    "hon": "hơn",
    "huong": "hương",
    "khong": "không",
    "la": "là",
    "lai": "lại",
    "lam": "lắm",
    "lan": "lần",
    "loai": "loại",
    "luong": "lượng",
    "minh": "mình",
    "moi": "mới",
    "mot": "một",
    "mua": "mua",
    "muc": "mức",
    "mui": "mùi",
    "muon": "muốn",
    "nay": "này",
    "nen": "nên",
    "ngay": "ngày",
    "nghi": "nghĩ",
    "nguoi": "người",
    "nhanh": "nhanh",
    "nhat": "nhất",
    "nhieu": "nhiều",
    "nhin": "nhìn",
    "nhung": "nhưng",
    "noi": "nói",
    "qua": "quá",
    "rat": "rất",
    "roi": "rồi",
    "san": "sản",
    "sao": "sao",
    "se": "sẽ",
    "su": "sự",
    "tam": "tạm",
    "thay": "thấy",
    "them": "thêm",
    "thi": "thì",
    "thich": "thích",
    "tot": "tốt",
    "toi": "tôi",
    "trai": "trải",
    "trang": "trắng",
    "truoc": "trước",
    "tuyet": "tuyệt",
    "va": "và",
    "van": "vẫn",
    "ve": "về",
    "vi": "vì",
    "voi": "với",
}

PHRASE_OVERRIDES = {
    "chat luong": "chất lượng",
    "cham soc": "chăm sóc",
    "chong nang": "chống nắng",
    "dich vu": "dịch vụ",
    "dong goi": "đóng gói",
    "dong sua": "dòng sữa",
    "don hang": "đơn hàng",
    "giai dap": "giải đáp",
    "giao hang": "giao hàng",
    "hai long": "hài lòng",
    "ho tro": "hỗ trợ",
    "khach hang": "khách hàng",
    "khong bi": "không bị",
    "loai da": "loại da",
    "moi nguoi": "mọi người",
    "mot so": "một số",
    "nguoi ban": "người bán",
    "phu hop": "phù hợp",
    "qua trinh": "quá trình",
    "san pham": "sản phẩm",
    "su chuyen nghiep": "sự chuyên nghiệp",
    "tang can": "tăng cân",
    "thai do": "thái độ",
    "thoi gian": "thời gian",
    "trai nghiem": "trải nghiệm",
    "van chuyen": "vận chuyển",
    "kcn": "kem chống nắng"
}

def strip_vietnamese_accents(text):
    text = unicodedata.normalize("NFD", str(text))
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D")

def build_diacritic_restore_map(texts):
    candidates = defaultdict(Counter)
    for text in texts.dropna().astype(str):
        for token in re.findall(r"\w+", text.lower(), flags=re.UNICODE):
            key = strip_vietnamese_accents(token)
            if key:
                candidates[key][token] += 1

    restore_map = {
        key: counter.most_common(1)[0][0]
        for key, counter in candidates.items()
    }
    restore_map.update(DIACRITIC_OVERRIDES)
    restore_map.update(SLANG_DICT)
    return restore_map

def preserve_case(original, restored):
    if original.isupper():
        return restored.upper()
    if original[:1].isupper():
        return restored.capitalize()
    return restored

def has_vietnamese_diacritics(token):
    return token != strip_vietnamese_accents(token)

def apply_phrase_overrides(text):
    restored = str(text)
    for phrase, replacement in sorted(PHRASE_OVERRIDES.items(), key=lambda item: len(item[0]), reverse=True):
        restored = re.sub(rf"\b{re.escape(phrase)}\b", replacement, restored)
    return restored

def restore_vietnamese_diacritics(text, restore_map):
    text = apply_phrase_overrides(text)
    tokens = re.findall(r"\w+|\s+|[^\w\s]", str(text), flags=re.UNICODE)
    restored_tokens = []
    for token in tokens:
        if re.fullmatch(r"\w+", token, flags=re.UNICODE):
            if has_vietnamese_diacritics(token.lower()):
                restored_tokens.append(token)
                continue
            key = strip_vietnamese_accents(token.lower())
            restored = restore_map.get(key, token)
            restored_tokens.append(preserve_case(token, restored))
        else:
            restored_tokens.append(token)
    return "".join(restored_tokens)

def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def normalize_slang(text):
    words = text.split()
    norm_words = []
    for word in words:
        norm_words.append(SLANG_DICT.get(word, word))
        # replace if it's a slang
    return " ".join(norm_words)

def normalize_repeated_characters(text):
    # Reduce expressive 3+ repeated chars, but keep valid doubles like "shipper".
    return re.sub(r"(.)\1{2,}", r"\1", text)

def normalize_text(text):
    text = normalize_unicode(text)
    text = normalize_slang(text)
    text = normalize_repeated_characters(text)
    text = normalize_whitespace(text)

    return text

accent_restore_map = build_diacritic_restore_map(df["review"])
print(f"Accent restoration dictionary size: {len(accent_restore_map):,} tokens")

Accent restoration dictionary size: 4,265 tokens


In [ ]:
# Bỏ stopword với các stopword được định nghĩa trong "vietnamese-stopwords.txt"


# Word_segmentation

In this step, we define what is a correct word to preserve the meaning of words. This step is critical because Vietnamese word is not word by word separated by spaces, but a combination, for example "hoc sinh" not "hoc" and "sinh". We have 2 libraries to help with this.

In [62]:
from underthesea import word_tokenize
from pyvi.ViTokenizer import tokenize

def segment_underthesea(text):
    return word_tokenize(text, format="text")
    # segment and return a string (format="text")

def segment_pyvi(text):
    return tokenize(text)

def segment_text(text, method="underthesea"):
    if method == "underthesea":
        return segment_underthesea(text)
    elif method == "pyvi":
        return segment_pyvi(text)
    else:
        raise ValueError("Invalid segmentation method")

In [63]:
df["segmented_review"] = df["normalized_review"].apply(segment_pyvi)
df_unaccented["segmented_review"] = df_unaccented["normalized_review"].apply(segment_pyvi)

df_all = pd.concat(
    [
        df.assign(source="shopee"),
        df_unaccented.assign(source="aug_unaccented"),
    ],
    ignore_index=True,
)

display(df_all[["source", "review", "normalized_review", "segmented_review", "label"]].head())
display(df_all[["source", "review", "normalized_review", "segmented_review", "label"]].tail())

,source,review,normalized_review,segmented_review,label
0,shopee,Hương vị:thom Chắc do bên giao hàng bị vỡ mấy...,hương vị thơm chắc độ bên giao hàng bị vỡ mấy gói,hương_vị thơm chắc độ bên giao hàng bị vỡ mấy gói,negative
1,shopee,Hương thơm:nhẹ nhàng Lợi ích:phục hồi cấp ẩm M...,hương thơm nhẹ nhàng lợi ích phục hồi cấp ẩm m...,hương thơm nhẹ_nhàng lợi_ích phục_hồi cấp ẩm m...,positive
2,shopee,Chất lượng sản phẩm:ok Đúng với mô tả:đúng Lầ...,chất lượng sản phẩm ok đúng với mô tả đúng lần...,chất_lượng sản_phẩm ok đúng với mô_tả đúng lần...,positive
3,shopee,Độ tuổi sử dụng:em bes Chất lượng sản phẩm:tot...,độ tuổi sử dụng em bes chất lượng sản phẩm tốt...,độ tuổi sử_dụng em bes chất_lượng sản_phẩm tốt...,positive
4,shopee,"Hương vị:Mix vị, Tím Mình mua 64 gói (32 gói ...",hương vị mix vị tím mình mua 64 gói 32 gói mix...,hương_vị mix vị tím mình mua 64 gói 32 gói mix...,negative


,source,review,normalized_review,segmented_review,label
10942,aug_unaccented,Kiem soat dau:toc bet da man bet Lam sach:cong...,kiểm soát đầu tóc bết da màn bết lắm sạch công...,kiểm_soát đầu tóc bết da màn bết lắm sạch công...,negative
10943,aug_unaccented,Truoc gio xai kcn khac ko sao nay boi thu loai...,trước giờ xài kcn khác không sao này bôi thử l...,trước giờ xài kcn khác không sao này bôi thử l...,negative
10944,aug_unaccented,Minh bi di ung rat nhieu sua tam nen chi dung ...,mình bị đi ủng rất nhiều sữa tạm nên chỉ dụng ...,mình bị đi ủng rất nhiều sữa tạm nên chỉ dụng ...,positive
10945,aug_unaccented,Ua sao thay co qua tang ma mo ra thi ko co sho...,ủa sao thấy có quá tặng mà mô ra thì không có ...,ủa sao thấy có quá tặng mà mô ra thì không có ...,negative
10946,aug_unaccented,Gia sp cao hon ca gia ngoai.du ap ma.ao gia,giá sản phẩm cao hơn cả giá ngoài đủ áp mà áo giá,giá sản_phẩm cao hơn cả giá ngoài đủ áp mà áo giá,negative


# Tokenize

TF-IDF is used for classical ML models. PhoBERT tokenization is used for transformer-based deep learning. This is where the ML and DL pipelines split: TF-IDF returns a sparse matrix, while PhoBERT tokenization returns PyTorch tensors.

In [64]:
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer

def tokenize_tfidf(tokenizer, texts):
    return tokenizer.fit_transform(texts)

def phobert_tokenizer():
    return AutoTokenizer.from_pretrained("vinai/phobert-base")

def tokenize_phobert(tokenizer, text):
    return tokenizer(
        text,
        padding="max_length", # force all sequences to have the same length
        truncation=True, # truncate if token too long
        max_length=128,
        return_tensors="pt", # return pytorch tensor
    )

In [65]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tokens_ML = tokenize_tfidf(tfidf, df_all["segmented_review"])
tokens_ML

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 400498 stored elements and shape (10947, 5000)>

In [66]:
phobert = phobert_tokenizer()
tokens_DL = df_all["segmented_review"].apply(
    lambda x: tokenize_phobert(phobert, x)
)
tokens_DL

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

d:\Tony\Career\ProjectNhom\Customer_sentiment_analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tony\.cache\huggingface\hub\models--vinai--phobert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

0        {'input_ids': [[tensor(0), tensor(4088), tenso...
1        {'input_ids': [[tensor(0), tensor(2263), tenso...
2        {'input_ids': [[tensor(0), tensor(390), tensor...
3        {'input_ids': [[tensor(0), tensor(378), tensor...
4        {'input_ids': [[tensor(0), tensor(4088), tenso...
                               ...                        
10942    {'input_ids': [[tensor(0), tensor(659), tensor...
10943    {'input_ids': [[tensor(0), tensor(71), tensor(...
10944    {'input_ids': [[tensor(0), tensor(68), tensor(...
10945    {'input_ids': [[tensor(0), tensor(19684), tens...
10946    {'input_ids': [[tensor(0), tensor(133), tensor...
Name: segmented_review, Length: 10947, dtype: object

In [69]:
feature_names = tfidf.get_feature_names_out()

row = tokens_ML[0]

for idx, value in zip(row.indices, row.data):
    print(feature_names[idx], value)

hương_vị 0.22279595036218866
thơm 0.14508758183307022
chắc 0.2208746244671727
độ 0.1595547501725289
bên 0.19443702939291055
giao 0.11361343482455204
hàng 0.09090806631388434
bị 0.12663406380987124
vỡ 0.21491193548157728
mấy 0.19209877225765812
gói 0.1928662236823223
hương_vị thơm 0.3378268545670653
chắc độ 0.34360427202936095
độ bên 0.3302380042993074
bên giao 0.3770988601925622
giao hàng 0.13819471449515722
hàng bị 0.2866197878080226
bị vỡ 0.2739841177116846


In [72]:
idx = 0

print("Raw:")
print(df_all["review"].iloc[idx])

print("\nSegmented:")
print(df_all["segmented_review"].iloc[idx])



Raw:
Hương vị:thom  Chắc do bên giao hàng bị vỡ mấy gói

Segmented:
hương_vị thơm chắc độ bên giao hàng bị vỡ mấy gói
